# EMPIEZA ROCHA

# Learning Urban Crimes Representation

## Detección de señales de disparidad en el reporte
Hipótesis operativa: zonas funcionalmente similares (cercanas en embedding
space) deberían exhibir patrones de reporte razonablemente próximos.
Aquellas que se desvían persistentemente son candidatas a discordancia.

Métodos:
  A) kNN regression — estimar perfil esperado por vecinos en embedding space
  B) Isolation Forest — detección de anomalías multivariada
  C) Local Outlier Factor (LOF) — anomalías basadas en densidad local
  D) Autoencoder de reconstrucción — error como señal de anomalía
  E) Score compuesto — integración de todos los métodos

Input:  embeddings_h3_optimal.csv, firmas_h3.csv, h3_metadata.csv,
        clusters_h3_optimal.csv

Output: disparidad_scores.csv — scores de disparidad por hexágono
        disparidad_zonas_candidatas.csv — zonas con señales persistentes
        mapa_disparidad.html — visualización geográfica

### Paquetes

In [36]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn

# visualización
import folium
import branca.colormap as cm
import h3 as h3lib

### Ejecución

#### Configuración de torch

In [37]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

Device: cuda


#### Cargar datos

In [38]:
embeddings = pd.read_csv("../data/results/embeddings_h3_optimal.csv", index_col='h3_id')
firmas = pd.read_csv("../data/auxiliar/firmas_h3.csv", index_col='h3_id')
metadata = pd.read_csv("../data/auxiliar/h3_metadata.csv", index_col='h3_id')
clusters = pd.read_csv("../data/results/clusters_h3_optimal.csv", index_col='h3_id')

emb_cols = [c for c in embeddings.columns if c.startswith('emb_')]
X_emb = embeddings[emb_cols].values
X_firmas = firmas.values
feature_names = firmas.columns.tolist()

print(f"Hexágonos: {len(firmas)}")
print(f"Embeddings: {X_emb.shape[1]}d")
print(f"Firmas: {X_firmas.shape[1]}d")

# Estandarizar firmas
scaler_firmas = StandardScaler()
X_firmas_std = scaler_firmas.fit_transform(X_firmas)

Hexágonos: 1061
Embeddings: 12d
Firmas: 45d


#### Implementación

In [39]:
# ============================================================================
# PASO 1: kNN REGRESSION — perfil esperado vs observado
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A: kNN regression")
print(f"{'='*80}")
print(f"  Estimando perfil esperado para cada zona basado en sus vecinos...")

# Para cada hexágono: encontrar los K vecinos más cercanos en embedding space,
# promediar sus firmas, y comparar con la firma observada.

K_NEIGHBORS = 10

knn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric='euclidean')  # +1 porque se incluye a sí mismo
knn_model.fit(X_emb)
distances, indices = knn_model.kneighbors(X_emb)

# Para cada hexágono, calcular perfil esperado (promedio de vecinos, excluyéndose)
perfiles_esperados = np.zeros_like(X_firmas_std)
for i in range(len(X_emb)):
    vecinos_idx = indices[i, 1:]  # excluir el propio punto (índice 0)
    perfiles_esperados[i] = X_firmas_std[vecinos_idx].mean(axis=0)

# Error de discrepancia por hexágono
discrepancia_knn = np.sqrt(np.mean((X_firmas_std - perfiles_esperados) ** 2, axis=1))

# ¿En qué dimensiones discrepa más cada hexágono?
discrepancia_por_dim = np.abs(X_firmas_std - perfiles_esperados)

print(f"  K vecinos: {K_NEIGHBORS}")
print(f"  Discrepancia media: {discrepancia_knn.mean():.4f}")
print(f"  Discrepancia P95: {np.percentile(discrepancia_knn, 95):.4f}")
print(f"  Discrepancia máxima: {discrepancia_knn.max():.4f}")


Método A: kNN regression
  Estimando perfil esperado para cada zona basado en sus vecinos...
  K vecinos: 10
  Discrepancia media: 0.7574
  Discrepancia P95: 1.6920
  Discrepancia máxima: 3.3662


# TERMINA ROCHA

# EMPIEZA ADOLF

In [40]:
# ============================================================================
# PASO 2: RANDOM FOREST — predicción del volumen esperado
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A2: Random forest — Volumen esperado vs observado")
print(f"{'='*80}")

# Usar embeddings para predecir intensidad (log registros)
# Un hexágono con mucho menos volumen del esperado dado su perfil → posible subreporte

y_intensidad = firmas['intensidad_log'].values

# Leave-one-out prediction con Random Forest
intensidad_predicha = np.zeros_like(y_intensidad)

for i in range(len(X_emb)):
    vecinos_idx = indices[i, 1:]  # K vecinos en embedding space
    X_train = X_emb[vecinos_idx]
    y_train = y_intensidad[vecinos_idx]
    
    # También incluir vecinos de vecinos para más datos de entrenamiento
    all_train_idx = set()
    for v in vecinos_idx:
        all_train_idx.update(indices[v, 1:].tolist())
    all_train_idx.discard(i)  # excluir el punto evaluado
    all_train_idx = list(all_train_idx)
    
    if len(all_train_idx) > 5:
        X_train = X_emb[all_train_idx]
        y_train = y_intensidad[all_train_idx]
    
    rf = RandomForestRegressor(n_estimators=50, max_depth=4, random_state=42)
    rf.fit(X_train, y_train)
    intensidad_predicha[i] = rf.predict(X_emb[i:i+1])[0]

residuo_intensidad = y_intensidad - intensidad_predicha  # negativo = menos de lo esperado
residuo_intensidad_abs = np.abs(residuo_intensidad)

print(f"  RMSE: {np.sqrt(mean_squared_error(y_intensidad, intensidad_predicha)):.3f}")
print(f"  Residuo medio: {residuo_intensidad.mean():.3f}")
print(f"  Residuo std: {residuo_intensidad.std():.3f}")

# Zonas con significativamente menos reporte del esperado
umbral_sub = residuo_intensidad.mean() - 1.5 * residuo_intensidad.std()
n_sub = (residuo_intensidad < umbral_sub).sum()
print(f"  Zonas con volumen < esperado (1.5σ): {n_sub}")


Método A2: Random forest — Volumen esperado vs observado
  RMSE: 0.782
  Residuo medio: -0.042
  Residuo std: 0.780
  Zonas con volumen < esperado (1.5σ): 87


In [41]:
# ============================================================================
# PASO 3: ISOLATION FOREST
# ============================================================================
print(f"\n{'='*80}")
print(f"Método B: Isolation Forest|")
print(f"{'='*80}")

# Sobre firmas estandarizadas — detecta hexágonos con perfiles atípicos
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.1,  # esperamos ~10% de anomalías
    random_state=42,
    max_features=0.8
)
iso_labels = iso_forest.fit_predict(X_firmas_std)
iso_scores = -iso_forest.score_samples(X_firmas_std)  # más alto = más anómalo

n_anomalas_iso = (iso_labels == -1).sum()
print(f"  Anomalías detectadas: {n_anomalas_iso} ({n_anomalas_iso/len(iso_labels)*100:.1f}%)")
print(f"  Score medio (normales): {iso_scores[iso_labels == 1].mean():.4f}")
print(f"  Score medio (anomalías): {iso_scores[iso_labels == -1].mean():.4f}")


Método B: Isolation Forest|
  Anomalías detectadas: 106 (10.0%)
  Score medio (normales): 0.3785
  Score medio (anomalías): 0.5166


In [42]:

# ============================================================================
# PASO 4: LOCAL OUTLIER FACTOR
# ============================================================================
print(f"\n{'='*80}")
print(f"Método C: Local Outlier Factor")
print(f"{'='*80}")

# LOF sobre embeddings — detecta zonas que son outliers respecto a sus vecinos
# en el espacio aprendido (no en el espacio original)
lof = LocalOutlierFactor(
    n_neighbors=15,
    contamination=0.1,
    metric='euclidean'
)
lof_labels = lof.fit_predict(X_emb)
lof_scores = -lof.negative_outlier_factor_  # más alto = más anómalo

n_anomalas_lof = (lof_labels == -1).sum()
print(f"  Anomalías detectadas: {n_anomalas_lof} ({n_anomalas_lof/len(lof_labels)*100:.1f}%)")
print(f"  LOF score medio (normales): {lof_scores[lof_labels == 1].mean():.4f}")
print(f"  LOF score medio (anomalías): {lof_scores[lof_labels == -1].mean():.4f}")


Método C: Local Outlier Factor
  Anomalías detectadas: 106 (10.0%)
  LOF score medio (normales): 1.0094
  LOF score medio (anomalías): 1.1337


# TERMINA ADOLF

# EMPIEZA ARA

In [43]:

# ============================================================================
# PASO 5: AUTOENCODER DE RECONSTRUCCIÓN (sobre firmas)
# ============================================================================
print(f"\n{'='*80}")
print(f"Método D: Autoencoder de reconstrucción")
print(f"{'='*80}")

class AnomalyAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 8),
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
        )
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

X_tensor = torch.FloatTensor(X_firmas_std)
dataset = torch.utils.data.TensorDataset(X_tensor, X_tensor)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

model_ae = AnomalyAutoencoder(X_firmas_std.shape[1])
optimizer = torch.optim.Adam(model_ae.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
criterion = nn.MSELoss()

best_loss = float('inf')
patience_counter = 0

#print(f"  Entrenando autoencoder de anomalías")
for epoch in range(300):
    model_ae.train()
    epoch_loss = 0
    for bx, _ in dataloader:
        optimizer.zero_grad()
        xr, z = model_ae(bx)
        loss = criterion(xr, bx)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(dataloader)
    scheduler.step(avg)
    if avg < best_loss:
        best_loss = avg
        patience_counter = 0
        best_state = model_ae.state_dict().copy()
    else:
        patience_counter += 1
    if patience_counter >= 50:
        print(f"  Early stopping epoch {epoch+1}")
        break

model_ae.load_state_dict(best_state)
model_ae.eval()
with torch.no_grad():
    xr, z = model_ae(X_tensor)
    recon_error = torch.mean((X_tensor - xr) ** 2, dim=1).numpy()

print(f"  Mejor loss: {best_loss:.6f}")
print(f"  Error de reconstrucción medio: {recon_error.mean():.4f}")
print(f"  Error de reconstrucción P95: {np.percentile(recon_error, 95):.4f}")



Método D: Autoencoder de reconstrucción
  Early stopping epoch 268
  Mejor loss: 0.507805
  Error de reconstrucción medio: 0.4300
  Error de reconstrucción P95: 1.6635


In [44]:

# ============================================================================
# PASO 6: SCORE COMPUESTO DE DISPARIDAD
# ============================================================================
print(f"\n{'='*80}")
print(f"Score compuesto de disparidad")
print(f"{'='*80}")

# Normalizar todos los scores a [0, 1]
scaler_01 = MinMaxScaler()

scores_raw = pd.DataFrame({
    'discrepancia_knn': discrepancia_knn,
    'residuo_intensidad': residuo_intensidad_abs,
    'iso_score': iso_scores,
    'lof_score': lof_scores,
    'recon_error': recon_error,
}, index=firmas.index)

scores_norm = pd.DataFrame(
    scaler_01.fit_transform(scores_raw),
    columns=[f'{c}_norm' for c in scores_raw.columns],
    index=firmas.index
)

# Score compuesto ponderado
# kNN y residuo intensidad pesan más (son los más interpretables para disparidad de reporte)
scores_norm['disparidad_score'] = (
    0.25 * scores_norm['discrepancia_knn_norm'] +
    0.30 * scores_norm['residuo_intensidad_norm'] +
    0.15 * scores_norm['iso_score_norm'] +
    0.15 * scores_norm['lof_score_norm'] +
    0.15 * scores_norm['recon_error_norm']
)

# Dirección del residuo (negativo = posible subreporte)
scores_norm['residuo_direccion'] = np.where(residuo_intensidad < 0, 'SUBREPORTE', 'SOBREREPORTE')
scores_norm['residuo_intensidad_raw'] = residuo_intensidad

# Clasificar nivel de disparidad
p75 = scores_norm['disparidad_score'].quantile(0.75)
p90 = scores_norm['disparidad_score'].quantile(0.90)
p95 = scores_norm['disparidad_score'].quantile(0.95)

def clasificar_disparidad(score):
    if score >= p95: return 'MUY ALTA'
    if score >= p90: return 'ALTA'
    if score >= p75: return 'MODERADA'
    return 'BAJA'

scores_norm['nivel_disparidad'] = scores_norm['disparidad_score'].apply(clasificar_disparidad)

print(f"  Distribución de niveles:")
for nivel, count in scores_norm['nivel_disparidad'].value_counts().items():
    print(f"    {nivel:15s} {count:>5d} ({count/len(scores_norm)*100:.1f}%)")


Score compuesto de disparidad
  Distribución de niveles:
    BAJA              795 (74.9%)
    MODERADA          159 (15.0%)
    MUY ALTA           54 (5.1%)
    ALTA               53 (5.0%)


In [45]:
# ============================================================================
# PASO 7: ANÁLISIS DE ZONAS CANDIDATAS
# ============================================================================
print(f"\n{'='*80}")
print(f"Zonas candidatas con alta disparidad")
print(f"{'='*80}")

# Unir con metadata
resultado = scores_norm.join(scores_raw).join(metadata).join(firmas[['intensidad_log', 'ratio_violencia']])
resultado['intensidad_real'] = np.expm1(resultado['intensidad_log']).astype(int)
resultado['cluster'] = clusters['cluster_kmeans']

# En qué dimensiones discrepan más las zonas de alta disparidad
delito_cols = [c for c in firmas.columns if c.startswith('delito_')]

# Top zonas con MUY ALTA disparidad
zonas_alta = resultado[resultado['nivel_disparidad'].isin(['MUY ALTA', 'ALTA'])].sort_values(
    'disparidad_score', ascending=False
)

print(f"\n  Zonas con disparidad ALTA o MUY ALTA: {len(zonas_alta)}")

for _, row in zonas_alta.head(20).iterrows():
    idx = firmas.index.get_loc(row.name)
    
    # Top dimensiones de discrepancia
    disc_dims = pd.Series(discrepancia_por_dim[idx], index=feature_names)
    top_disc = disc_dims.nlargest(3)
    dims_str = ', '.join([f"{n.replace('delito_','').replace('_',' ')}: {v:.2f}" 
                          for n, v in top_disc.items()])
    
    print(f"\n  ▸ {row['alcaldia_dominante']} — {row['colonia_dominante']}")
    print(f"    Score: {row['disparidad_score']:.3f} ({row['nivel_disparidad']})")
    print(f"    Registros: {row['intensidad_real']:,} | Violencia: {row['ratio_violencia']:.3f}")
    print(f"    Residuo intensidad: {row['residuo_intensidad_raw']:.2f} ({row['residuo_direccion']})")
    print(f"    Cluster: {int(row['cluster'])} | Discrepancia kNN: {row['discrepancia_knn']:.3f}")
    print(f"    Mayores discrepancias: {dims_str}")


Zonas candidatas con alta disparidad

  Zonas con disparidad ALTA o MUY ALTA: 107

  ▸ MIGUEL HIDALGO — REFINERÍA 18 DE MARZO
    Score: 0.772 (MUY ALTA)
    Registros: 47 | Violencia: 0.125
    Residuo intensidad: -3.81 (SUBREPORTE)
    Cluster: 9 | Discrepancia kNN: 2.734
    Mayores discrepancias: delitos de servidores publicos: 6.74, portacion de armas: 6.59, dia wednesday: 6.14

  ▸ MIGUEL HIDALGO — BOSQUE DE CHAPULTEPEC III SECCIÓN
    Score: 0.697 (MUY ALTA)
    Registros: 31 | Violencia: 0.226
    Residuo intensidad: -2.98 (SUBREPORTE)
    Cluster: 12 | Discrepancia kNN: 2.816
    Mayores discrepancias: dia tuesday: 8.00, hora tarde: 7.81, dia thursday: 7.50

  ▸ XOCHIMILCO — PARQUE ECOLOGICO DE XOCHIMILCO
    Score: 0.657 (MUY ALTA)
    Registros: 36 | Violencia: 0.216
    Residuo intensidad: -3.84 (SUBREPORTE)
    Cluster: 6 | Discrepancia kNN: 2.044
    Mayores discrepancias: dia saturday: 6.70, trim Q2: 4.67, trim Q4: 4.29

  ▸ CUAJIMALPA DE MORELOS — LA PILA
    Score: 0.

In [46]:
# ============================================================================
# PASO 8: ANÁLISIS POR ALCALDÍA
# ============================================================================
print(f"\n{'='*80}")
print(f"Disparidad por alcaldía")
print(f"{'='*80}")

disp_alcaldia = resultado.groupby('alcaldia_dominante').agg({
    'disparidad_score': ['mean', 'median', 'max'],
    'residuo_intensidad_raw': 'mean',
    'nivel_disparidad': lambda x: (x.isin(['ALTA', 'MUY ALTA'])).sum(),
}).round(3)

disp_alcaldia.columns = ['score_medio', 'score_mediana', 'score_max', 
                           'residuo_medio', 'n_zonas_alta_disp']
disp_alcaldia = disp_alcaldia.sort_values('score_medio', ascending=False)

print(f"\n  {'Alcaldía':30s} {'Score medio':>12s} {'Mediana':>8s} {'Max':>6s} {'Residuo':>8s} {'Zonas alta':>11s}")
print(f"  {'-'*80}")
for alc, row in disp_alcaldia.iterrows():
    print(f"  {alc:30s} {row['score_medio']:>10.3f} {row['score_mediana']:>8.3f} "
          f"{row['score_max']:>6.3f} {row['residuo_medio']:>8.2f} {int(row['n_zonas_alta_disp']):>8d}")



Disparidad por alcaldía

  Alcaldía                        Score medio  Mediana    Max  Residuo  Zonas alta
  --------------------------------------------------------------------------------
  MILPA ALTA                          0.277    0.246  0.629     0.01       15
  CUAJIMALPA DE MORELOS               0.248    0.203  0.644    -0.06       13
  XOCHIMILCO                          0.203    0.167  0.657    -0.12       14
  MIGUEL HIDALGO                      0.195    0.143  0.772    -0.17        9
  TLAHUAC                             0.188    0.138  0.582    -0.12        9
  TLALPAN                             0.188    0.163  0.546    -0.01       18
  VENUSTIANO CARRANZA                 0.144    0.097  0.525    -0.05        3
  ALVARO OBREGON                      0.140    0.120  0.544     0.03        3
  AZCAPOTZALCO                        0.132    0.117  0.422    -0.08        3
  IZTAPALAPA                          0.131    0.094  0.605    -0.08       11
  GUSTAVO A. MADERO         

In [47]:
# ============================================================================
# PASO 9: ANÁLISIS POR CLUSTER
# ============================================================================
print(f"\n{'='*80}")
print(f"Disparidad por cluster")
print(f"{'='*80}")

disp_cluster = resultado.groupby('cluster').agg({
    'disparidad_score': ['mean', 'std'],
    'residuo_intensidad_raw': ['mean', 'std'],
    'nivel_disparidad': lambda x: (x.isin(['ALTA', 'MUY ALTA'])).sum(),
    'intensidad_real': 'median',
}).round(3)

disp_cluster.columns = ['score_medio', 'score_std', 'residuo_medio', 'residuo_std',
                          'n_alta_disp', 'registros_mediana']
disp_cluster = disp_cluster.sort_values('score_medio', ascending=False)

print(f"\n  {'Cluster':>7s} {'Score':>7s} {'±σ':>6s} {'Residuo':>8s} {'±σ':>6s} {'Alta disp':>10s} {'Registros':>10s}")
print(f"  {'-'*60}")
for cl, row in disp_cluster.iterrows():
    print(f"  {int(cl):>7d} {row['score_medio']:>7.3f} {row['score_std']:>6.3f} "
          f"{row['residuo_medio']:>8.2f} {row['residuo_std']:>6.2f} "
          f"{int(row['n_alta_disp']):>8d} {int(row['registros_mediana']):>10,}")



Disparidad por cluster

  Cluster   Score     ±σ  Residuo     ±σ  Alta disp  Registros
  ------------------------------------------------------------
        8   0.278  0.115     0.01   0.94       25        174
       13   0.263  0.121    -0.01   0.91       12        222
       14   0.194  0.115    -0.08   0.77        7        571
        5   0.189  0.126    -0.12   0.88       11        713
       12   0.173  0.130    -0.05   0.86        6        843
        3   0.171  0.115    -0.00   0.83        8        737
       10   0.162  0.108    -0.10   0.90        7        959
       11   0.147  0.074     0.02   0.73        3      1,446
        1   0.136  0.087    -0.05   0.73        6      1,609
        4   0.136  0.103    -0.08   0.72        8      2,049
        9   0.134  0.108    -0.09   0.86        4      2,311
        0   0.124  0.089     0.04   0.72        4      4,191
        2   0.110  0.095    -0.03   0.70        2      2,670
        7   0.106  0.080    -0.03   0.49        2      4

In [48]:
# ============================================================================
# PASO 10: CONCORDANCIA ENTRE MÉTODOS
# ============================================================================
print(f"\n{'='*80}")
print(f"Concordancia entre métodos")
print(f"{'='*80}")

# Top 10% por cada método
n_top = int(len(firmas) * 0.1)
top_knn = set(scores_raw['discrepancia_knn'].nlargest(n_top).index)
top_iso = set(scores_raw['iso_score'].nlargest(n_top).index)
top_lof = set(scores_raw['lof_score'].nlargest(n_top).index)
top_ae = set(scores_raw['recon_error'].nlargest(n_top).index)
top_rf = set(scores_raw['residuo_intensidad'].nlargest(n_top).index)

metodos = {'kNN': top_knn, 'IsoForest': top_iso, 'LOF': top_lof, 
           'AutoEnc': top_ae, 'RF_residuo': top_rf}

# Zonas detectadas por múltiples métodos
from collections import Counter
detecciones = Counter()
for h3_id in firmas.index:
    count = sum(1 for nombre, top_set in metodos.items() if h3_id in top_set)
    detecciones[h3_id] = count

zonas_multi = {h: c for h, c in detecciones.items() if c >= 3}
print(f"  Zonas detectadas por ≥3 métodos: {len(zonas_multi)}")
print(f"  Zonas detectadas por ≥4 métodos: {sum(1 for c in zonas_multi.values() if c >= 4)}")
print(f"  Zonas detectadas por 5 métodos: {sum(1 for c in zonas_multi.values() if c == 5)}")

resultado['n_metodos_detectan'] = [detecciones[h] for h in resultado.index]


Concordancia entre métodos
  Zonas detectadas por ≥3 métodos: 96
  Zonas detectadas por ≥4 métodos: 52
  Zonas detectadas por 5 métodos: 13


In [ ]:
# ============================================================================
# PASO 11: EXPORTAR
# ============================================================================

# Scores completos
export_cols = ['disparidad_score', 'nivel_disparidad', 'residuo_direccion',
               'discrepancia_knn', 'residuo_intensidad_raw', 'iso_score', 'lof_score',
               'recon_error', 'n_metodos_detectan',
               'alcaldia_dominante', 'colonia_dominante', 'cluster',
               'intensidad_real', 'ratio_violencia']
resultado[export_cols].to_csv('../data/results/disparidad_scores.csv', encoding='utf-8-sig')
print(f"../data/results/disparidad_scores.csv ({len(resultado)} hexágonos)")

# Zonas candidatas (alta + muy alta)
zonas_export = resultado[resultado['nivel_disparidad'].isin(['ALTA', 'MUY ALTA'])][export_cols]
zonas_export = zonas_export.sort_values('disparidad_score', ascending=False)
zonas_export.to_csv('../data/results/disparidad_zonas_candidatas.csv', encoding='utf-8-sig')
print(f"../data/results/disparidad_zonas_candidatas.csv ({len(zonas_export)} zonas)")

# Resumen por alcaldía
disp_alcaldia.to_csv('../data/results/disparidad_por_alcaldia.csv', encoding='utf-8-sig')
print(f"../data/results/disparidad_por_alcaldia.csv")


../data/results/disparidad_scores.csv (1061 hexágonos)
../data/results/disparidad_zonas_candidatas.csv (107 zonas)
../data/results/disparidad_por_alcaldia.csv


# TERMINA ARA

# EMPIEZA LUISEN

In [50]:
# ============================================================================
# PASO 12: MAPA DE DISPARIDAD
# ============================================================================

CDMX_CENTER = [19.38, -99.14]

m = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB dark_matter')

# Colormap: verde (baja disparidad) → amarillo → rojo (alta disparidad)
vmin_d = resultado['disparidad_score'].quantile(0.05)
vmax_d = resultado['disparidad_score'].quantile(0.95)
colormap_disp = cm.LinearColormap(
    colors=['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027'],
    vmin=vmin_d, vmax=vmax_d,
    caption='Score de Disparidad'
)

for h3_id, row in resultado.iterrows():
    boundary = h3lib.cell_to_boundary(h3_id)
    polygon = [[lat, lng] for lat, lng in boundary]
    score = row['disparidad_score']
    color = colormap_disp(min(max(score, vmin_d), vmax_d))
    
    popup_html = f"""
    <b>Score disparidad: {score:.3f}</b> ({row['nivel_disparidad']})<br>
    <b>{row['alcaldia_dominante']}</b> — {row['colonia_dominante']}<br>
    Registros: {int(row['intensidad_real']):,}<br>
    Residuo: {row['residuo_intensidad_raw']:.2f} ({row['residuo_direccion']})<br>
    Violencia: {row['ratio_violencia']:.3f}<br>
    Cluster: {int(row['cluster'])}<br>
    Métodos que detectan: {int(row['n_metodos_detectan'])}/5
    """
    
    weight = 2.5 if row['nivel_disparidad'] in ['ALTA', 'MUY ALTA'] else 0.8
    opacity = 0.85 if row['nivel_disparidad'] in ['ALTA', 'MUY ALTA'] else 0.6
    
    folium.Polygon(
        locations=polygon,
        color=color,
        weight=weight,
        fill=True,
        fill_color=color,
        fill_opacity=opacity,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"Disparidad: {score:.3f} | {row['alcaldia_dominante']}"
    ).add_to(m)

colormap_disp.add_to(m)
m.save('../visualizations/mapa_disparidad.html')
print(f"../visualizations/mapa_disparidad.html")


../visualizations/mapa_disparidad.html


# TERMINA LUISEN